In [52]:
# Re-import necessary modules after code execution environment reset
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np



In [53]:
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_aer.primitives import EstimatorV2
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [54]:
# General-purpose quantum gate for forget/input/output
def create_quantum_gate(n_qubits=2):
    input_params = ParameterVector("x", n_qubits)
    weight_params = ParameterVector("θ", n_qubits)

    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(input_params[i], i)
        qc.rz(weight_params[i], i)

    estimator = EstimatorV2()
    qnn = EstimatorQNN(
        circuit=qc,
        input_params=input_params,
        weight_params=weight_params,
        estimator=estimator
    )

    return TorchConnector(qnn)

In [55]:
# Define updated LSTM cell with quantum forget, input, and output gates
class FullQuantumLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size,
                 q_forget_gate, q_input_gate, q_output_gate):
        super().__init__()
        self.hidden_size = hidden_size
        self.q_forget_gate = q_forget_gate
        self.q_input_gate = q_input_gate
        self.q_output_gate = q_output_gate

        self.forget_encoder = nn.Linear(input_size + hidden_size, 2)
        self.input_encoder = nn.Linear(input_size + hidden_size, 2)
        self.output_encoder = nn.Linear(input_size + hidden_size, 2)
        self.candidate = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(self, x_t, h_prev, c_prev):
        combined = torch.cat([x_t, h_prev], dim=-1)

        f_input = torch.tanh(self.forget_encoder(combined))
        f_t = torch.sigmoid(self.q_forget_gate(f_input))

        i_input = torch.tanh(self.input_encoder(combined))
        i_t = torch.sigmoid(self.q_input_gate(i_input))

        c_tilde = torch.tanh(self.candidate(combined))
        c_t = f_t * c_prev + i_t * c_tilde

        o_input = torch.tanh(self.output_encoder(combined))
        o_t = torch.sigmoid(self.q_output_gate(o_input))

        h_t = o_t * torch.tanh(c_t)
        return h_t, c_t


In [56]:
# Wrap cell in full sequence model
class FullQuantumLSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size,
                 q_forget, q_input, q_output):
        super().__init__()
        self.cell = FullQuantumLSTMCell(input_size, hidden_size, q_forget, q_input, q_output)
        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        h_t = torch.zeros(batch_size, self.cell.hidden_size)
        c_t = torch.zeros(batch_size, self.cell.hidden_size)

        for t in range(seq_len):
            h_t, c_t = self.cell(x[:, t], h_t, c_t)

        return self.output_layer(h_t)

In [57]:
# Dummy data generation
def generate_dummy_data(seq_len=10, num_samples=256):
    X = []
    y = []
    for _ in range(num_samples):
        phase = np.random.rand() * 2 * np.pi
        x = np.linspace(0, 2 * np.pi, seq_len + 1) + phase
        series = np.sin(x)
        X.append(series[:-1])
        y.append(series[-1])
    X = np.array(X)
    y = np.array(y)
    return torch.tensor(X, dtype=torch.float32).unsqueeze(-1), torch.tensor(y, dtype=torch.float32).unsqueeze(-1)

In [58]:
# Create quantum gates for the 3 components
q_forget = create_quantum_gate()
q_input = create_quantum_gate()
q_output = create_quantum_gate()

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.
No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.
No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


In [59]:
# Initialize model
model_full_quantum = FullQuantumLSTMModel(input_size=1, hidden_size=4,
                                          q_forget=q_forget,
                                          q_input=q_input,
                                          q_output=q_output)

In [60]:
# Training setup
criterion = nn.MSELoss()
optimizer = optim.Adam(model_full_quantum.parameters(), lr=0.01)

In [61]:
# Generate data
X_train, y_train = generate_dummy_data()

In [62]:
# Training loop
for epoch in range(10):
    optimizer.zero_grad()
    output = model_full_quantum(X_train)
    loss = criterion(output, y_train)
    loss.backward()
    optimizer.step()

loss.item()  # Return final loss

0.5129987597465515

In [ ]:
# Re-import necessary modules after code reset
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit_aer.primitives import Estimator
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Generate dummy sine wave data
def generate_dummy_data(seq_len=10, num_samples=256):
    X = []
    y = []
    for _ in range(num_samples):
        phase = np.random.rand() * 2 * np.pi
        x = np.linspace(0, 2 * np.pi, seq_len + 1) + phase
        series = np.sin(x)
        X.append(series[:-1])
        y.append(series[-1])
    X = np.array(X)
    y = np.array(y)
    return torch.tensor(X, dtype=torch.float32).unsqueeze(-1), torch.tensor(y, dtype=torch.float32).unsqueeze(-1)

X_train, y_train = generate_dummy_data()

# Create entangled quantum gate
def create_entangled_quantum_gate(n_qubits=2):
    input_params = ParameterVector("x", n_qubits)
    weight_params = ParameterVector("θ", n_qubits)

    qc = QuantumCircuit(n_qubits)
    for i in range(n_qubits):
        qc.ry(input_params[i], i)
        qc.rz(weight_params[i], i)
    for i in range(n_qubits - 1):
        qc.cx(i, i + 1)

    estimator = Estimator()
    qnn = EstimatorQNN(
        circuit=qc,
        input_params=input_params,
        weight_params=weight_params,
        estimator=estimator
    )

    return TorchConnector(qnn)

# Define updated LSTM cell
class FullQuantumLSTMCell(nn.Module):
    def __init__(self, input_size, hidden_size,
                 q_forget_gate, q_input_gate, q_output_gate):
        super().__init__()
        self.hidden_size = hidden_size
        self.q_forget_gate = q_forget_gate
        self.q_input_gate = q_input_gate
        self.q_output_gate = q_output_gate

        self.forget_encoder = nn.Linear(input_size + hidden_size, 2)
        self.input_encoder = nn.Linear(input_size + hidden_size, 2)
        self.output_encoder = nn.Linear(input_size + hidden_size, 2)
        self.candidate = nn.Linear(input_size + hidden_size, hidden_size)

    def forward(self, x_t, h_prev, c_prev):
        combined = torch.cat([x_t, h_prev], dim=-1)

        f_input = torch.tanh(self.forget_encoder(combined))
        f_t = torch.sigmoid(self.q_forget_gate(f_input))

        i_input = torch.tanh(self.input_encoder(combined))
        i_t = torch.sigmoid(self.q_input_gate(i_input))

        c_tilde = torch.tanh(self.candidate(combined))
        c_t = f_t * c_prev + i_t * c_tilde

        o_input = torch.tanh(self.output_encoder(combined))
        o_t = torch.sigmoid(self.q_output_gate(o_input))

        h_t = o_t * torch.tanh(c_t)
        return h_t, c_t

# Full improved model
class ImprovedQuantumLSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size,
                 q_forget, q_input, q_output):
        super().__init__()
        self.cell = FullQuantumLSTMCell(input_size, hidden_size, q_forget, q_input, q_output)
        self.output_layer = nn.Linear(hidden_size, 1)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        h_t = torch.zeros(batch_size, self.cell.hidden_size)
        c_t = torch.zeros(batch_size, self.cell.hidden_size)

        for t in range(seq_len):
            h_t, c_t = self.cell(x[:, t], h_t, c_t)

        return self.output_layer(h_t)

# Build model with improved components
q_forget = create_entangled_quantum_gate()
q_input = create_entangled_quantum_gate()
q_output = create_entangled_quantum_gate()

model_improved = ImprovedQuantumLSTMModel(input_size=1, hidden_size=16,
                                          q_forget=q_forget,
                                          q_input=q_input,
                                          q_output=q_output)

# Train model
criterion = nn.MSELoss()
optimizer = optim.Adam(model_improved.parameters(), lr=0.01)

for epoch in range(50):  # More epochs
    optimizer.zero_grad()
    output = model_improved(X_train)
    loss = criterion(output, y_train)
    loss.backward()
    optimizer.step()

loss.item()


/tmp/ipykernel_15570/1928290603.py:107: DeprecationWarning: Estimator has been deprecated as of Aer 0.15, please use EstimatorV2 instead.
  q_forget = create_entangled_quantum_gate()
/tmp/ipykernel_15570/1928290603.py:107: DeprecationWarning: Option approximation=False is deprecated as of qiskit-aer 0.13. It will be removed no earlier than 3 months after the release date. Instead, use BackendEstimator from qiskit.primitives.
  q_forget = create_entangled_quantum_gate()
/tmp/ipykernel_15570/1928290603.py:46: DeprecationWarning: V1 Primitives are deprecated as of qiskit-machine-learning 0.8.0 and will be removed no sooner than 4 months after the release date. Use V2 primitives for continued compatibility and support.
  qnn = EstimatorQNN(
/tmp/ipykernel_15570/1928290603.py:108: DeprecationWarning: Estimator has been deprecated as of Aer 0.15, please use EstimatorV2 instead.
  q_input = create_entangled_quantum_gate()
/tmp/ipykernel_15570/1928290603.py:108: DeprecationWarning: Option appr